In [1]:
import pandas as pd 
import numpy as np
import re
import seaborn as sns

import scikit_posthocs as sp
from scipy.stats import friedmanchisquare
from matplotlib import pyplot as plt



In [2]:
import sys 
sys.path.append('../source')

In [3]:


def extraire_donnees_vers_tableau(nom_fichier):
    with open(nom_fichier, 'r') as f:
        contenu = f.read()


    blocs = contenu.strip().split("Algorithm:")[1:]
    
    liste_resultats = []
    
    for bloc in blocs:
        lignes = bloc.strip().split('\n')
        nom_algo = lignes[0].strip()
        
        parametres = {}
        for ligne in lignes[1:]:
            if ':' in ligne:
                cle, valeur = ligne.split(':', 1)
                cle = cle.strip()
                valeur = valeur.strip()
                
                if cle not in ['File name', 'ROC', 'Running time']:
                    parametres[cle] = valeur
        
        str_params = ", ".join([f"{k}: {v}" for k, v in parametres.items()])
        
        liste_resultats.append({
            "Algorithm": nom_algo,
            "Parameters": str_params
        })

    df = pd.DataFrame(liste_resultats)
    
    df = df.drop_duplicates().reset_index(drop=True)
    
    return df

try:
    nom_du_fichier = '../result'
    tableau_final = extraire_donnees_vers_tableau(nom_du_fichier)
    
    #print("Tableau des Algorithmes et Paramètres :")
    print(tableau_final.to_markdown(index=False))
    
    #tableau_final.to_latex("results/parametres_algos.tex", index=False)
    
except FileNotFoundError:
    print("Erreur : Le fichier n'a pas été trouvé. Vérifiez le nom du fichier.")

| Algorithm   | Parameters                                                                |
|:------------|:--------------------------------------------------------------------------|
| LODA        | num_bins: 10, num_random_cuts: 100                                        |
| xStream     | num_components: 100, n_chains: 100, depth: 25, window_size: 25            |
| HSTree      | window_size: 100, num_trees: 25, max_depth: 15                            |
| RSHash      | sampling_points: 1000, decay: 0.015, num_components: 100, num_hash_fns: 1 |
| IForestASD  | window_size: 2048                                                         |
| LODA        | num_bins: 20, num_random_cuts: 50                                         |
| LODA        | num_bins: 5, num_random_cuts: 200                                         |
| xStream     | num_components: 50, n_chains: 50, depth: 15, window_size: 50              |
| xStream     | num_components: 200, n_chains: 200, depth: 35, window_size: 10  

In [4]:
file_path = "../result"

with open(file_path, 'r') as file:
    content = file.read()

In [3]:

def extraire_donnees_completes(nom_fichier):
    try:
        with open(nom_fichier, 'r', encoding='utf-8') as f:
            contenu = f.read()
    except FileNotFoundError:
        return "Erreur : Le fichier '{}' est introuvable.".format(nom_fichier)

    blocs = re.split(r'\n(?=Algorithm:)', contenu.strip())
    
    liste_resultats = []

    for bloc in blocs:
        lignes = bloc.strip().split('\n')
        info_entree = {}
        
        for ligne in lignes:
            if ':' in ligne:
                cle, valeur = ligne.split(':', 1)
                cle = cle.strip()
                valeur = valeur.strip()
       
                try:
                    if '.' in valeur:
                        valeur = float(valeur)
                    else:
                        valeur = int(valeur)
                except ValueError:
                    pass 
                
                info_entree[cle] = valeur
        
        if info_entree:
            liste_resultats.append(info_entree)

    # Création du DataFrame
    df = pd.DataFrame(liste_resultats)
    

    col_file = 'File name'
    col_roc = 'ROC'
    col_time = 'Running time'

    # S'assurer que les colonnes critiques existent
    for col in [col_file, col_roc, col_time]:
        if col not in df.columns:
            df[col] = None

    # 4. Trouver le meilleur algo par fichier
    # Tri : ROC descendant, puis Temps ascendant
    df_sorted = df.sort_values(by=[col_file, col_roc, col_time], 
                               ascending=[True, False, True])
    
    # On garde la meilleure ligne pour chaque fichier
    meilleurs = df_sorted.groupby(col_file).first().reset_index()
    
    return meilleurs


In [6]:
import re

def generer_rapport_final(nom_fichier):
    try:
        with open(nom_fichier, 'r', encoding='utf-8') as f:
            contenu = f.read()
    except FileNotFoundError:
        return f"Erreur : Le fichier '{nom_fichier}' est introuvable."

    blocs = re.split(r'\n(?=Algorithm:)', contenu.strip())
    
    data = []
    for bloc in blocs:
        algo = re.search(r'Algorithm:\s*(.*)', bloc)
        file = re.search(r'File name:\s*(.*)', bloc)
        roc = re.search(r'ROC:\s*([\d.]+)', bloc)
        time = re.search(r'Running time:\s*([\d.]+)', bloc)

        h_params = re.findall(r'^([\w_]+):\s*([\d.]+)', bloc, re.MULTILINE)
        params_dict = {k: v for k, v in h_params if k not in ['ROC', 'Running_time']}

        if algo and file and roc:
            data.append({
                'Algorithm': algo.group(1).strip(),
                'Dataset': file.group(1).strip(),
                'ROC': float(roc.group(1)),
                'Time': float(time.group(1)) if time else 0.0,
                'Params': params_dict
            })

    df = pd.DataFrame(data)


    df = df.sort_values(by=['Dataset', 'ROC', 'Time'], ascending=[True, False, True])
    
    df_time_mean = (
        df.groupby(['Dataset', 'Algorithm'])['Time']
        .mean()
        .reset_index()
    )

    pivot_time = df_time_mean.pivot(
        index='Dataset',
        columns='Algorithm',
            values='Time'
    )
        
        

    df['Display'] = df.apply(lambda x: f"{x['ROC']:.4f} ({x['Time']:.1f}s)", axis=1)

    # Pivotage
    pivot_roc = df.pivot_table(index='Dataset', columns='Algorithm', values='ROC', aggfunc='first')
    pivot_display = df.pivot_table(index='Dataset', columns='Algorithm', values='Display', aggfunc='first')
    

    def souligner_meilleur(row_name):
        rocs = pivot_roc.loc[row_name]
        displays = pivot_display.loc[row_name]
        max_roc = rocs.max()
        
        new_row = []
        for r, d in zip(rocs, displays):
            # On met en gras si c'est le max (et pas NaN)
            if r == max_roc and pd.notna(r):
                new_row.append(f"**{d}**")
            else:
                new_row.append(str(d) if pd.notna(d) else "N/A")
        return new_row

    # Construction du tableau final stylis
    final_data = []
    for dataset in pivot_roc.index:
        final_data.append(souligner_meilleur(dataset))

    df_final = pd.DataFrame(final_data, index=pivot_roc.index, columns=pivot_roc.columns)
    
    
    return df_final, pivot_roc, pivot_time


In [7]:

nom_fichier = "../result"
tableau, pivot_roc , pivot_time= generer_rapport_final(nom_fichier)

if isinstance(tableau, pd.DataFrame):
    print(tableau.to_markdown())
    
   
else:
    print(tableau)

| Dataset                    | HSTree            | IForestASD          | LODA            | RSHash          | xStream              |
|:---------------------------|:------------------|:--------------------|:----------------|:----------------|:---------------------|
| COIL20_mix_sudden          | 0.9266 (1208.4s)  | **0.9565 (257.7s)** | 0.5001 (133.6s) | 0.9167 (178.0s) | 0.9521 (1829.8s)     |
| Ionosphere_shake_gradual   | 0.7318 (79.1s)    | **0.9823 (145.8s)** | 0.5000 (613.3s) | 0.7329 (218.6s) | 0.9233 (821.7s)      |
| Ionosphere_shake_sudden_2  | 0.7312 (98.8s)    | **0.9786 (158.9s)** | 0.5000 (192.9s) | 0.7343 (294.1s) | 0.9204 (1028.3s)     |
| S1                         | 0.8771 (632.9s)   | **0.9831 (39.1s)**  | 0.7246 (83.0s)  | 0.9682 (19.1s)  | 0.9560 (400.8s)      |
| S2                         | **0.6152 (0.7s)** | 0.6059 (25.2s)      | 0.5002 (32.9s)  | 0.5595 (28.4s)  | 0.5430 (122.2s)      |
| WOBC_shake_gradual_2       | 0.7305 (109.2s)   | 0.9278 (159.8s)     | 0.5

In [8]:
#pivot_time.to_csv("../results/pivot_time_sota.csv")

In [9]:
pivot_roc

Algorithm,HSTree,IForestASD,LODA,RSHash,xStream
Dataset,,,,,
COIL20_mix_sudden,0.9266,0.9565,0.5001,0.9167,0.9521
Ionosphere_shake_gradual,0.7318,0.9823,0.5000,0.7329,0.9233
Ionosphere_shake_sudden_2,0.7312,0.9786,0.5000,0.7343,0.9204
S1,0.8771,0.9831,0.7246,0.9682,0.9560
S2,0.6152,0.6059,0.5002,0.5595,0.5430
WOBC_shake_gradual_2,0.7305,0.9278,0.5000,0.7342,0.9771
WOBC_shake_sudden,0.7353,0.9432,0.5001,0.7361,0.9794
dermatology_shake_gradual,0.7447,0.9896,0.5000,0.7316,0.9950
dermatology_shake_sudden_2,0.7321,0.9850,0.5000,0.7130,0.9959


In [11]:
#pivot_roc.to_csv("results/pivot_roc_sota.csv", index=False)

In [4]:

nom_du_fichier = "../result"
df_final = extraire_donnees_completes(nom_du_fichier)


print(df_final.to_string(index=False))

#df_final.to_csv("synthese_performances.csv", index=False)

                 File name  Algorithm    ROC  Running time  num_bins  num_random_cuts  num_components  n_chains  depth  window_size  num_trees  max_depth  sampling_points  decay  num_hash_fns
         COIL20_mix_sudden IForestASD 0.9565      257.6550      20.0             50.0            50.0      50.0   15.0       1024.0       35.0       20.0           2000.0  0.020           1.0
  Ionosphere_shake_gradual IForestASD 0.9823      145.8133       5.0            200.0            50.0      50.0   15.0       1024.0       25.0       15.0           2000.0  0.020           1.0
 Ionosphere_shake_sudden_2 IForestASD 0.9786      158.8626      20.0             50.0            50.0      50.0   15.0       1024.0       25.0       15.0           2000.0  0.020           1.0
                        S1 IForestASD 0.9831       39.1305      20.0             50.0            50.0      50.0   15.0        200.0       35.0       20.0            500.0  0.010           2.0
                        S2     HSTree 0.

In [6]:
df_final.isna().sum()

File name          0
Algorithm          0
ROC                0
Running time       0
num_bins           0
num_random_cuts    0
num_components     0
n_chains           0
depth              0
window_size        0
num_trees          0
max_depth          0
sampling_points    0
decay              0
num_hash_fns       0
dtype: int64

In [ ]:

def generer_rapport_final(nom_fichier):
    try:
        with open(nom_fichier, 'r', encoding='utf-8') as f:
            contenu = f.read()
    except FileNotFoundError:
        return f"Erreur : Le fichier '{nom_fichier}' est introuvable."

    blocs = re.split(r'\n(?=Algorithm:)', contenu.strip())
    data = []
    for bloc in blocs:
        algo = re.search(r'Algorithm:\s*(.*)', bloc)
        file = re.search(r'File name:\s*(.*)', bloc)
        roc = re.search(r'ROC:\s*([\d.]+)', bloc)
        time = re.search(r'Running time:\s*([\d.]+)', bloc)

        if algo and file and roc:
            data.append({
                'Algorithm': algo.group(1).strip(),
                'Dataset': file.group(1).strip(),
                'ROC': float(roc.group(1)),
                'Time': float(time.group(1)) if time else 0.0
            })

    df = pd.DataFrame(data)

    stats = df.groupby(['Dataset', 'Algorithm']).agg({
        'ROC': ['mean', 'std'],
        'Time': ['mean', 'std']
    })
    
    stats.columns = ['ROC_mean', 'ROC_std', 'Time_mean', 'Time_std']
    stats = stats.reset_index()

    pivot_roc_mean = stats.pivot(index='Dataset', columns='Algorithm', values='ROC_mean')
    pivot_time_mean = stats.pivot(index='Dataset', columns='Algorithm', values='Time_mean')

   
    def format_cell(row):
        # Formatage : 0.XXX ± 0.XXX
        mean = row['ROC_mean']
        std = row['ROC_std']
        if pd.isna(std):
            return f"{mean:.4f}"
        return f"{mean:.4f} ± {std:.4f}"

    stats['Display'] = stats.apply(format_cell, axis=1)
    pivot_display = stats.pivot(index='Dataset', columns='Algorithm', values='Display')

    # 5. Mise en gras du meilleur ROC par Dataset
    def souligner_meilleur(row_name):
        means = pivot_roc_mean.loc[row_name]
        displays = pivot_display.loc[row_name]
        max_val = means.max()
        
        new_row = []
        for m, d in zip(means, displays):
            if m == max_val and pd.notna(m):
                new_row.append(f"\\textbf{{{d}}}") 
            else:
                new_row.append(str(d) if pd.notna(d) else "N/A")
        return new_row

    final_data = [souligner_meilleur(ds) for ds in pivot_display.index]
    df_final = pd.DataFrame(final_data, index=pivot_display.index, columns=pivot_display.columns)
    
    return df_final, pivot_roc_mean, pivot_time_mean


In [44]:
import re
import pandas as pd
import numpy as np

def generer_rapport_final(nom_fichier):
    try:
        with open(nom_fichier, 'r', encoding='utf-8') as f:
            contenu = f.read()
    except FileNotFoundError:
        return f"Erreur : Le fichier '{nom_fichier}' est introuvable."

    blocs = re.split(r'\n(?=Algorithm:)', contenu.strip())
    data = []
    for bloc in blocs:
        algo = re.search(r'Algorithm:\s*(.*)', bloc)
        file = re.search(r'File name:\s*(.*)', bloc)
        roc = re.search(r'ROC:\s*([\d.]+)', bloc)
        time = re.search(r'Running time:\s*([\d.]+)', bloc)

        if algo and file and roc:
            data.append({
                'Algorithm': algo.group(1).strip(),
                'Dataset': file.group(1).strip(),
                'ROC': float(roc.group(1)),
                'Time': float(time.group(1)) if time else 0.0
            })

    df = pd.DataFrame(data)


    stats = df.groupby(['Dataset', 'Algorithm']).agg({
        'ROC': ['mean', 'std'],
        'Time': ['mean', 'std']
    })
    stats.columns = ['ROC_mean', 'ROC_std', 'Time_mean', 'Time_std']
    stats = stats.reset_index()

    # 3. Création des pivots numériques (Retours 2 et 3)
    pivot_roc_mean = stats.pivot(index='Dataset', columns='Algorithm', values='ROC_mean')
    pivot_time_mean = stats.pivot(index='Dataset', columns='Algorithm', values='Time_mean')

    print(stats)
    def format_val(mean, std, precision=4):
        if pd.isna(std) or std == 0:
            return f"{mean:.{precision}f}"
        return f"{mean:.{precision}f} ± {std:.{precision}f}"

    stats['ROC_display'] = stats.apply(lambda x: format_val(x['ROC_mean'], x['ROC_std'], 4), axis=1)
    stats['Time_display'] = stats.apply(lambda x: format_val(x['Time_mean'], x['Time_std'], 2), axis=1)

    pivot_roc_txt = stats.pivot(index='Dataset', columns='Algorithm', values='ROC_display')
    pivot_time_txt = stats.pivot(index='Dataset', columns='Algorithm', values='Time_display')

    def styliser(pivot_val, pivot_txt, mode='max'):
        final_rows = []
        for dataset in pivot_val.index:
            vals = pivot_val.loc[dataset]
            texts = pivot_txt.loc[dataset]
            
            target = vals.max() if mode == 'max' else vals.min()
            
            new_row = []
            for v, t in zip(vals, texts):
                if v == target and pd.notna(v):
                    new_row.append(f"\\textbf{{{t}}}")
                else:
                    new_row.append(str(t) if pd.notna(t) else "N/A")
            final_rows.append(new_row)
        return pd.DataFrame(final_rows, index=pivot_val.index, columns=pivot_val.columns)


    df_final = styliser(pivot_roc_mean, pivot_roc_txt, mode='max')
    df_final_time_test = styliser(pivot_time_mean, pivot_time_txt, mode='min')
    
    return df_final, pivot_roc_mean, pivot_time_mean, df_final_time_test


In [45]:
df_str, df_roc, df_time, res_time = generer_rapport_final("../result")

                Dataset   Algorithm  ROC_mean   ROC_std    Time_mean  \
0     COIL20_mix_sudden      HSTree  0.924800  0.003032   423.754533   
1     COIL20_mix_sudden  IForestASD  0.917700  0.050651  1898.944300   
2     COIL20_mix_sudden        LODA  0.500033  0.000058   315.603167   
3     COIL20_mix_sudden      RSHash  0.905467  0.009922    97.024633   
4     COIL20_mix_sudden     xStream  0.910000  0.059761  6500.906300   
..                  ...         ...       ...       ...          ...   
90  wine_shake_sudden_2      HSTree  0.742067  0.009952   818.313433   
91  wine_shake_sudden_2  IForestASD  0.978433  0.006958   290.948933   
92  wine_shake_sudden_2        LODA  0.500000  0.000000   434.127133   
93  wine_shake_sudden_2      RSHash  0.741633  0.001850   167.245000   
94  wine_shake_sudden_2     xStream  0.954950  0.011950  7788.153700   

       Time_std  
0    680.061428  
1   2714.485462  
2    209.997019  
3     70.959147  
4   5733.296513  
..          ...  
90  1326.

In [46]:
res_time

Algorithm,HSTree,IForestASD,LODA,RSHash,xStream
Dataset,,,,,
COIL20_mix_sudden,423.75 ± 680.06,1898.94 ± 2714.49,315.60 ± 210.00,\textbf{97.02 ± 70.96},6500.91 ± 5733.30
Ionosphere_shake_gradual,564.91 ± 906.68,\textbf{289.64 ± 171.10},358.54 ± 232.93,883.35 ± 1263.44,5788.90 ± 7024.74
Ionosphere_shake_sudden_2,707.11 ± 1135.36,302.73 ± 172.01,447.54 ± 290.02,\textbf{164.75 ± 113.27},7433.18 ± 9057.82
S1,317.37 ± 446.20,\textbf{37.95 ± 1.67},192.72 ± 155.10,44.33 ± 35.72,4157.02 ± 5312.16
S2,123.27 ± 173.31,22.28 ± 4.14,81.25 ± 68.34,\textbf{18.16 ± 14.51},899.90 ± 1099.84
WOBC_shake_gradual_2,958.27 ± 1558.38,298.95 ± 162.28,479.61 ± 307.20,\textbf{194.33 ± 146.82},14901.68 ± 19145.35
WOBC_shake_sudden,504.03 ± 817.64,270.49 ± 158.06,261.30 ± 168.22,\textbf{101.44 ± 69.56},6194.31 ± 7709.79
dermatology_shake_gradual,564.27 ± 905.77,284.82 ± 166.40,358.59 ± 232.97,\textbf{138.36 ± 102.36},8122.84 ± 6463.06
dermatology_shake_sudden_2,705.78 ± 1132.62,293.61 ± 166.32,2483.60 ± 3804.07,\textbf{162.25 ± 117.54},7438.82 ± 9056.23


In [41]:
res_time

Algorithm,HSTree,IForestASD,LODA,RSHash,xStream
Dataset,,,,,
COIL20_mix_sudden,423.75 ± 680.06,1898.94 ± 2714.49,315.60 ± 210.00,\textbf{97.02 ± 70.96},6500.91 ± 5733.30
Ionosphere_shake_gradual,564.91 ± 906.68,\textbf{289.64 ± 171.10},358.54 ± 232.93,883.35 ± 1263.44,5788.90 ± 7024.74
Ionosphere_shake_sudden_2,707.11 ± 1135.36,302.73 ± 172.01,447.54 ± 290.02,\textbf{164.75 ± 113.27},7433.18 ± 9057.82
S1,317.37 ± 446.20,\textbf{37.95 ± 1.67},192.72 ± 155.10,44.33 ± 35.72,4157.02 ± 5312.16
S2,123.27 ± 173.31,22.28 ± 4.14,81.25 ± 68.34,\textbf{18.16 ± 14.51},899.90 ± 1099.84
WOBC_shake_gradual_2,958.27 ± 1558.38,298.95 ± 162.28,479.61 ± 307.20,\textbf{194.33 ± 146.82},14901.68 ± 19145.35
WOBC_shake_sudden,504.03 ± 817.64,270.49 ± 158.06,261.30 ± 168.22,\textbf{101.44 ± 69.56},6194.31 ± 7709.79
dermatology_shake_gradual,564.27 ± 905.77,284.82 ± 166.40,358.59 ± 232.97,\textbf{138.36 ± 102.36},8122.84 ± 6463.06
dermatology_shake_sudden_2,705.78 ± 1132.62,293.61 ± 166.32,2483.60 ± 3804.07,\textbf{162.25 ± 117.54},7438.82 ± 9056.23


In [24]:
df_str, df_roc, df_time = generer_rapport_final("../result")

In [29]:
df_str.to_csv('../results/data/result_sota_text.csv', index=True)

In [30]:
df_roc.to_csv('../results/data/result_sota__roc_roc.csv', index=True)

In [31]:
df_time.to_csv('../results/data/result_sota__roc_time.csv', index=True)

In [28]:
df_roc

Algorithm,HSTree,IForestASD,LODA,RSHash,xStream
Dataset,,,,,
COIL20_mix_sudden,0.924800,0.917700,0.500033,0.905467,0.910000
Ionosphere_shake_gradual,0.730833,0.974367,0.499200,0.732567,0.913050
Ionosphere_shake_sudden_2,0.729500,0.970033,0.500000,0.733800,0.915350
S1,0.875100,0.980200,0.612300,0.963900,0.919650
S2,0.572950,0.596000,0.500100,0.549150,0.507250
WOBC_shake_gradual_2,0.728400,0.920533,0.500000,0.733200,0.951100
WOBC_shake_sudden,0.726900,0.940300,0.500067,0.733433,0.952200
dermatology_shake_gradual,0.743133,0.979500,0.500000,0.730033,0.988533
dermatology_shake_sudden_2,0.731333,0.979033,0.500000,0.710067,0.987900
